## 1. Configuração e carregamento da base de inferência

## Objetivo deste notebook

Este notebook tem como objetivo analisar a base de inferência (`inference.csv`) e verificar se ela é compatível com a base histórica (`history.csv`) para aplicação do modelo de churn.

A base de inferência representa os clientes atuais da Terra Signal que serão pontuados pelo modelo. Como o modelo foi treinado na base histórica, precisamos garantir que:

1. As duas bases possuem as mesmas variáveis.
2. As categorias são compatíveis.
3. As distribuições das variáveis são similares.
4. Os missing values são identificados e tratados antes da modelagem.

In [0]:
%pip install seaborn

In [0]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações de visualização
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Carregamento da base de inferência
file_path = "/Workspace/projeto_databricks_churn/Hackathon_Terra_Signal/data/raw/inference.csv"
df_inference = pd.read_csv(file_path)

# Criar uma cópia para preparação e análise
df_inference_clean = df_inference.copy()

display(df_inference_clean.head())

## 2. Preparação básica
Vamos aplicar os mesmos tratamentos iniciais utilizados na base histórica: padronizar PhoneService e converter tenure para numérico.

In [0]:
# Padronização de PhoneService
df_inference_clean["PhoneService"] = (
    df_inference_clean["PhoneService"].str.strip().str.capitalize()
)

# Conversão de tenure para numérico
df_inference_clean["tenure"] = pd.to_numeric(
    df_inference_clean["tenure"].replace("unknown", np.nan),
    errors="coerce"
)

## 3. Estrutura e qualidade dos dados

In [0]:
# Dimensões da base
print("Quantidade de linhas e colunas:", df_inference_clean.shape)

# Tipos de dados
print("\n=== TIPOS DE DADOS ===")

data_types = pd.DataFrame({
    "Variável": df_inference_clean.columns,
    "Tipo de dado": df_inference_clean.dtypes.values
})

display(data_types)

# Valores ausentes
print("\n=== VALORES AUSENTES ===")

missing_summary = pd.DataFrame({
    "Variável": df_inference_clean.columns,
    "Valores ausentes": df_inference_clean.isna().sum().values,
    "Percentual (%)": (
        df_inference_clean.isna().mean().values * 100
    ).round(2)
})

display(
    missing_summary[
        missing_summary["Valores ausentes"] > 0
    ]
)

# Registros duplicados
print(
    "\nLinhas completamente duplicadas:",
    df_inference_clean.duplicated().sum()
)

print(
    "CustomerIDs duplicados:",
    df_inference_clean["customerID"].duplicated().sum()
)

#### Interpretação da qualidade dos dados

A base de inferência possui 1.409 registros e 22 colunas. As principais observações são:

- **Não há registros duplicados** nem `customerIDs` repetidos.
- **Valores ausentes**: encontramos missing em apenas duas variáveis:
  - `tenure`: 4 registros (0,28%)
  - `TotalCharges`: 64 registros (4,54%)
- **Não há valores `"unknown"`** nas variáveis de texto.

Essa baixa taxa de missing é positiva e indica que a base está apta para a modelagem, com poucos tratamentos necessários.

In [0]:
# Verificar valores unknown
unknown_counts = {}

for coluna in df_inference_clean.select_dtypes(
    include=["object", "string"]
).columns:

    quantidade = (
        df_inference_clean[coluna]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("unknown")
        .sum()
    )

    if quantidade > 0:
        unknown_counts[coluna] = quantidade

# Criar tabela de resultados
unknown_summary = pd.DataFrame(
    list(unknown_counts.items()),
    columns=["Variável", "Quantidade de unknown"]
)

print("=== VALORES UNKNOWN ===")

if unknown_summary.empty:
    print("Nenhum valor 'unknown' encontrado.")
else:
    display(unknown_summary)

## 4. Distribuição das variáveis categóricas

In [0]:
# Identificar variáveis categóricas
categorical_cols = df_inference_clean.select_dtypes(
    include=["object", "string"]
).columns.tolist()

# Excluir identificador e texto livre
excluded_cols = ["customerID", "CustomerFeedback", "Churn"]

categorical_cols = [
    col for col in categorical_cols
    if col not in excluded_cols
]

# Consolidar frequências e percentuais
categorical_summary = []

for col in categorical_cols:

    counts = df_inference_clean[col].value_counts(dropna=False)
    percentages = (
        df_inference_clean[col]
        .value_counts(normalize=True, dropna=False) * 100
    )

    for category in counts.index:
        categorical_summary.append({
            "Variável": col,
            "Categoria": str(category),
            "Quantidade": counts[category],
            "Percentual (%)": round(percentages[category], 2)
        })

categorical_summary = pd.DataFrame(categorical_summary)

display(categorical_summary)

### Principais características das variáveis categóricas

A distribuição das categorias na base de inferência é bastante similar à observada na base histórica. Alguns destaques:

- **Contratos mensais (`Month-to-month`)** representam a maioria dos clientes (55%), o que é esperado e condiz com o perfil de maior risco de churn.
- **Fibra óptica (`Fiber optic`)** é a modalidade de internet mais frequente (44%), seguida por DSL (35%) e ausência de internet (21%).
- **`Electronic check`** é a forma de pagamento mais comum (34%), modalidade que historicamente apresenta maior taxa de churn.
- A maioria dos clientes não possui dependentes (70%) e está em **PaperlessBilling** (60%).

## 5. Distribuição das variáveis numéricas
Vamos utilizar as mesmas estatísticas descritivas da base histórica.

In [0]:
# Identificar variáveis numéricas
# SeniorCitizen é binária (0/1), portanto categorica, não numérica
numeric_cols = df_inference_clean.select_dtypes(
    include=np.number
).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != "SeniorCitizen"]

# Estatísticas descritivas
numeric_summary = (
    df_inference_clean[numeric_cols]
    .describe()
    .T
    .round(2)
    .reset_index()
    .rename(columns={"index": "Variável"})
)

display(numeric_summary)

#### Principais características das variáveis numéricas

As estatísticas descritivas mostram um perfil de clientes semelhante ao da base histórica:

- **tenure**: média de 32,5 meses, mediana de 28 meses. Há clientes desde o primeiro mês até 72 meses.
- **MonthlyCharges**: média de R$ 65,37, com mediana de R$ 70,40. A dispersão é alta (desvio padrão ~R$ 30).
- **TotalCharges**: média de R$ 2.306, mas com mediana de R$ 1.412, indicando assimetria positiva — poucos clientes com valores acumulados muito altos.
- **MonthlyIncome**: média de R$ 5.146, com distribuição relativamente simétrica.

A variável `SeniorCitizen` foi removida desta análise porque, apesar de numérica (0/1), representa uma categoria binária.

## 6. Comparação entre history e inference
Esta é a parte mais importante. Para garantir que a comparação funcione mesmo em notebooks separados, vamos carregar novamente a base histórica e aplicar os mesmos tratamentos básicos.

In [0]:
# Carregar a base histórica
df_history_compare = pd.read_csv(
    "/Volumes/workspace/default/churn_data/history.csv"
)

# Padronizar PhoneService
df_history_compare["PhoneService"] = (
    df_history_compare["PhoneService"].str.strip().str.capitalize()
)

# Converter tenure para numérico
df_history_compare["tenure"] = pd.to_numeric(
    df_history_compare["tenure"].replace("unknown", np.nan),
    errors="coerce"
)

### 6.1 Comparação da estrutura

In [0]:
# Comparar dimensões
structure_comparison = pd.DataFrame({
    "History": df_history_compare.shape,
    "Inference": df_inference_clean.shape
}, index=["Quantidade de linhas", "Quantidade de colunas"])

display(structure_comparison)

### 6.2 Comparação dos valores ausentes

In [0]:
# Comparar valores ausentes

# Identificar variáveis comuns às duas bases
common_cols = [
    col for col in df_history_compare.columns
    if col in df_inference_clean.columns
]

missing_comparison = pd.DataFrame({
    "Variável": common_cols,
    "History": df_history_compare[common_cols].isna().sum().values,
    "Inference": df_inference_clean[common_cols].isna().sum().values,
    "History (%)": (
        df_history_compare[common_cols].isna().mean().values * 100
    ).round(2),
    "Inference (%)": (
        df_inference_clean[common_cols].isna().mean().values * 100
    ).round(2)
})

display(missing_comparison)

In [0]:
# Comparar a quantidade de valores ausentes por variável
for coluna in common_cols:

    missing_history = df_history_compare[coluna].isna().sum()
    missing_inference = df_inference_clean[coluna].isna().sum()

    if missing_history > 0 or missing_inference > 0:

        print(f"\n=== {coluna} ===")

        print("History - índices dos registros ausentes:")
        print(df_history_compare.index[
            df_history_compare[coluna].isna()
        ].tolist())

        print("Inference - índices dos registros ausentes:")
        print(df_inference_clean.index[
            df_inference_clean[coluna].isna()
        ].tolist())

In [0]:
# Relação entre valores ausentes em tenure e TotalCharges

for nome, df in [
    ("History", df_history_compare),
    ("Inference", df_inference_clean)
]:

    print(f"\n=== {nome} ===")

    missing_pattern = pd.crosstab(
        df["tenure"].isna(),
        df["TotalCharges"].isna()
    )

    missing_pattern.index = ["tenure preenchido", "tenure ausente"]
    missing_pattern.columns = [
        "TotalCharges preenchido",
        "TotalCharges ausente"
    ]

    display(missing_pattern)

In [0]:
ids_history = set(df_history_compare["customerID"].dropna())
ids_inference = set(df_inference_clean["customerID"].dropna())

ids_comuns = ids_history.intersection(ids_inference)

print("IDs presentes nas duas bases:", len(ids_comuns))

### 6.3 Comparação das categorias
Aqui vamos comparar os percentuais de cada categoria nas duas bases.

In [0]:
# Variáveis categóricas presentes nas duas bases
# SeniorCitizen é binária (0/1), mas deve ser tratada como categorica
categorical_compare = [
    col for col in categorical_cols
    if col in df_history_compare.columns
    and col in df_inference_clean.columns
]

# Excluir variáveis que não devem entrar na comparação
categorical_compare = [
    col for col in categorical_compare
    if col not in ["customerID", "CustomerFeedback", "Churn"]
]

# Adicionar SeniorCitizen como categorica para comparacao
if "SeniorCitizen" not in categorical_compare:
    categorical_compare.append("SeniorCitizen")

category_comparison = []

for col in categorical_compare:

    history_pct = (
        df_history_compare[col]
        .value_counts(normalize=True, dropna=False) * 100
    )

    inference_pct = (
        df_inference_clean[col]
        .value_counts(normalize=True, dropna=False) * 100
    )

    categories = history_pct.index.union(inference_pct.index)

    for category in categories:
        category_comparison.append({
            "Variável": col,
            "Categoria": str(category),
            "History (%)": round(history_pct.get(category, 0), 2),
            "Inference (%)": round(inference_pct.get(category, 0), 2)
        })

category_comparison = pd.DataFrame(category_comparison)

category_comparison["Diferença (p.p.)"] = (
    category_comparison["Inference (%)"] -
    category_comparison["History (%)"]
).round(2)

display(category_comparison)

#### Interpretação da comparação de categorias

As diferenças entre history e inference são pequenas para a maioria das variáveis, geralmente abaixo de 3 pontos percentuais. Isso é um bom sinal: indica que a base de inferência não sofreu grandes mudanças de composição em relação ao período histórico.

Principais observações:
- **PhoneService**: pequeno aumento de clientes com telefone na inference (91,55% vs 90,01%).
- **StreamingTV**: leve redução de clientes com o serviço na inference.
- **TechSupport**: leve aumento de clientes com suporte técnico.

Nenhuma diferença parece grande o suficiente para comprometer a aplicabilidade do modelo.

### 6.4 Comparação das variáveis numéricas
Vamos comparar as estatísticas das variáveis numéricas nas duas bases.

In [0]:
# Variáveis numéricas presentes nas duas bases
numeric_compare = [
    col for col in df_history_compare.select_dtypes(include=np.number).columns
    if col in df_inference_clean.select_dtypes(include=np.number).columns
    and col != "SeniorCitizen"
]

# Estatísticas da base histórica
history_numeric = (
    df_history_compare[numeric_compare]
    .describe()
    .T
    .round(2)
    .add_prefix("History_")
)

# Estatísticas da base de inferência
inference_numeric = (
    df_inference_clean[numeric_compare]
    .describe()
    .T
    .round(2)
    .add_prefix("Inference_")
)

# Junta lado a lado
numeric_comparison = (
    history_numeric
    .join(inference_numeric)
    .reset_index()
    .rename(columns={"index": "Variável"})
)

# Reordena as colunas para ficar count, mean, std... comparando base a base
colunas_ordenadas = ["Variável"]
for metric in ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]:
    colunas_ordenadas.extend([
        f"History_{metric}",
        f"Inference_{metric}"
    ])

numeric_comparison = numeric_comparison[colunas_ordenadas]

display(numeric_comparison)

### Interpretação da comparação das variáveis numéricas

As estatísticas descritivas das variáveis numéricas são muito similares entre history e inference:

- **tenure**: média praticamente igual (~32,5 meses).
- **MonthlyCharges**: média ligeiramente maior na inference (R$ 65,37 vs R$ 64,61), mas dentro da variação esperada.
- **TotalCharges**: média e mediana próximas entre as bases.
- **MonthlyIncome**: distribuição muito similar.

Essa estabilidade reforça que o modelo treinado na history deve ter performance semelhante quando aplicado na inference.

## 6.5 Verificar compatibilidade das colunas
Por fim, vamos verificar se as duas bases possuem as mesmas variáveis, desconsiderando Churn, que é a variável-alvo e não deve estar na base de inferência.

In [0]:
# Colunas presentes somente em History
only_history = sorted(
    set(df_history_compare.columns) -
    set(df_inference_clean.columns) -
    {"Churn"}
)

# Colunas presentes somente em Inference
only_inference = sorted(
    set(df_inference_clean.columns) -
    set(df_history_compare.columns)
)

print("Colunas exclusivas de History:", only_history)
print("Colunas exclusivas de Inference:", only_inference)

## 6.6 Verificação de categorias novas na inference

Antes de aplicar o modelo, precisamos garantir que a base de inference não contenha categorias que não existiam na base histórica. Se houver categorias novas, o modelo pode quebrar ou produzir resultados incertos.

In [0]:
# Verifica se ha categorias na inference que nao existem no history
problemas_categoria = []

for col in categorical_compare:
    categorias_history = set(df_history_compare[col].dropna().astype(str).unique())
    categorias_inference = set(df_inference_clean[col].dropna().astype(str).unique())
    
    novas_categorias = categorias_inference - categorias_history
    
    if novas_categorias:
        problemas_categoria.append({
            'variavel': col,
            'categorias_novas': list(novas_categorias)
        })

if problemas_categoria:
    print('ATENCAO: categorias novas encontradas na inference:')
    for p in problemas_categoria:
        print(f"  {p['variavel']}: {p['categorias_novas']}")
else:
    print('OK: nenhuma categoria nova encontrada na inference.')
    print('Todas as categorias da inference estao presentes no history.')

#### Interpretação da verificação de categorias

A ausência de categorias novas na inference é um resultado importante. Significa que o modelo não encontrará valores inesperados nas variáveis categóricas durante a predição, reduzindo o risco de erros no scoring.

## 6.7 Visualização comparativa das distribuições

Comparar as distribuições das variáveis numéricas entre history e inference ajuda a identificar visualmente se há drift.

In [0]:
# Histogramas comparativos
variaveis_plot = ['tenure', 'MonthlyCharges', 'TotalCharges', 'MonthlyIncome']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

cor_history = "#4C9AA8"
cor_inference = "#E07A5F"
cor_cinza = "#BBBBBB"

for i, var in enumerate(variaveis_plot):
    ax = axes[i]
    
    hist_values = pd.to_numeric(df_history_compare[var], errors='coerce').dropna()
    inf_values = pd.to_numeric(df_inference_clean[var], errors='coerce').dropna()
    
    ax.hist(hist_values, bins=30, alpha=0.6, label='History', color=cor_history, density=True, edgecolor='white')
    ax.hist(inf_values, bins=30, alpha=0.6, label='Inference', color=cor_inference, density=True, edgecolor='white')
    
    # Todas as bordas em cinza
    for spine in ax.spines.values():
        spine.set_color(cor_cinza)
    
    # Marcas em cinza, numeros em preto
    ax.tick_params(axis='both', which='both', color=cor_cinza, labelcolor='black')
    
    # Grid sutil horizontal
    ax.grid(True, axis='y', color=cor_cinza, alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    ax.set_title(f'Distribuicao de {var}', fontsize=13, pad=12)
    ax.set_xlabel(var, fontsize=10)
    ax.set_ylabel('Densidade', fontsize=10)
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()

#### Interpretação dos histogramas

Os histogramas mostram que as distribuições das variáveis numéricas são visualmente muito similares entre history e inference. Não há mudanças abruptas de formato que indiquem drift significativo.

Pontos de atenção:
- **tenure**: concentração de clientes nos primeiros meses em ambas as bases.
- **MonthlyCharges**: pico em valores mais baixos, refletindo clientes com planos básicos ou sem internet.
- **TotalCharges**: distribuição assimétrica, com maior concentração em valores baixos.

In [0]:
# Box plots comparativos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

cor_history = "#4C9AA8"
cor_inference = "#E07A5F"
cor_cinza = "#BBBBBB"
palette = {"History": cor_history, "Inference": cor_inference}

for i, var in enumerate(variaveis_plot):
    ax = axes[i]
    
    hist_values = pd.to_numeric(df_history_compare[var], errors='coerce').dropna()
    inf_values = pd.to_numeric(df_inference_clean[var], errors='coerce').dropna()
    
    df_box = pd.DataFrame({
        var: pd.concat([hist_values, inf_values], ignore_index=True),
        'Base': ['History'] * len(hist_values) + ['Inference'] * len(inf_values)
    })
    
    box = sns.boxplot(data=df_box, x='Base', y=var, hue='Base', palette=palette, legend=False, ax=ax)
    
    # Transparencia
    for patch in box.artists:
        patch.set_alpha(0.6)
    
    # Estilo padronizado
    for spine in ax.spines.values():
        spine.set_color(cor_cinza)
    ax.tick_params(axis='both', which='both', color=cor_cinza, labelcolor='black')
    ax.grid(True, axis='y', color=cor_cinza, alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    ax.set_title(f'{var} por Base', fontsize=13, pad=12)
    ax.set_xlabel('Base', fontsize=10)
    ax.set_ylabel(var, fontsize=10)

plt.tight_layout()
plt.show()

#### Interpretação dos box plots

Os box plots confirmam a similaridade entre as distribuições. A mediana e a dispersão das variáveis numéricas são próximas entre history e inference, o que reforça a compatibilidade das bases.

## 6.8 Plano de tratamento dos missing values na inference

Na base de inference, identificamos missing values em duas variáveis:

| Variável | Missing na inference | Estratégia para o modelo |
|---|---|---|
| `tenure` | 4 registros (0,28%) | Como `tenure_numeric` entra no modelo via spline, precisamos tratar. Opções: remover os 4 clientes ou imputar com a mediana do treino. |
| `TotalCharges` | 64 registros (4,54%) | No nosso modelo final, `TotalCharges` foi removido por multicolinearidade. Portanto, não afeta a predição. |

**Recomendação**: imputar os 4 missing de `tenure_numeric` com a mediana da base de treino, garantindo que todos os 1.409 clientes recebam score.

In [0]:
# Exemplo de como sera feita a imputacao na modelagem
mediana_tenure_treino = df_history_compare['tenure'].median()

print(f'Mediana de tenure na history: {mediana_tenure_treino}')
print(f'Clientes com tenure missing na inference: {df_inference_clean["tenure"].isna().sum()}')

# Aplica imputacao (apenas demonstracao; a imputacao final deve ser feita no notebook de modelagem)
df_inference_demo = df_inference_clean.copy()
df_inference_demo['tenure_imputado'] = df_inference_demo['tenure'].fillna(mediana_tenure_treino)

print(f'Após imputacao: {df_inference_demo["tenure_imputado"].isna().sum()} missing')
df_inference_demo[['customerID', 'tenure', 'tenure_imputado']].head(10)

## 7. Conclusão da compatibilidade entre History e Inference

#### Pontos positivos
- As bases possuem as **mesmas colunas** (exceto `Churn`, que é o target e só existe na history).
- **Não há IDs em comum** entre history e inference, ou seja, não há vazamento de informação.
- **Não há categorias novas** na inference em relação ao history.
- As distribuições das variáveis categóricas e numéricas são **muito similares** entre as duas bases.
- As diferenças de percentuais entre categorias são pequenas (geralmente abaixo de 3 pontos percentuais).

#### Pontos de atenção
- Existem **4 missing values em `tenure`** na inference que precisam ser tratados antes da modelagem.
- Existem **64 missing values em `TotalCharges`** na inference, mas essa variável não entra no nosso modelo final.

#### Veredito
**A base de inference é compatível com a base history para aplicação do modelo de churn.** As diferenças observadas são pequenas e não devem comprometer a performance do modelo. O único tratamento necessário é a imputação dos 4 valores ausentes de `tenure`.